# Okanka Mini — 04: evaluation + export to the Hub ("okanka-mini")

Settings: GPU T4 (x1 is enough) or even CPU, Internet On.
1. pulls the SFT checkpoint, 2. runs a set of Turkish chat prompts, 3. computes validation bpb on the held-out
FineWeb-2 test shard, 4. exports weights + tokenizer + LICENSE/NOTICE + model card to `<HF_USERNAME>/okanka-mini`.

`MAKE_PUBLIC = True` makes the final model repo public (needed if the Space should load it without a token).

In [ ]:
# ======================= OKANKA MINI CONFIG (edit if needed) =======================
REPO_URL   = "https://github.com/CHANGE_ME/okanka-mini.git"   # GitHub repo with this code (public, or add Kaggle secret GITHUB_TOKEN)
BRANCH     = "main"
DEPTH      = 16          # model depth (d16 ~ 235M "scaling" params). Use 12 for a faster/cheaper run.
SEQ_LEN    = 1024        # context length used for pretraining on T4
DEVICE_BS  = 4 if DEPTH >= 14 else 8   # per-GPU micro batch (power of 2; halve it if CUDA OOM)
DATA_RATIO = 12          # tokens : scaling-params ratio (nanochat compute-optimal ~10-12; Chinchilla 20)
# Hugging Face repos (created automatically, private):
#   <HF_USERNAME>/okanka-mini-data         (dataset: pretraining shards + SFT jsonl)
#   <HF_USERNAME>/okanka-mini-checkpoints  (model: resumable training checkpoints)
#   <HF_USERNAME>/okanka-mini              (model: final public export, notebook 04)
# ===================================================================================

In [ ]:
import os, sys, subprocess, shutil
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name, default=None):
    try:
        return secrets.get_secret(name)
    except Exception:
        return default

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Add a Kaggle secret named HF_TOKEN (HF write token): Add-ons -> Secrets"
HF_USERNAME = secret("HF_USERNAME")
if not HF_USERNAME:
    from huggingface_hub import whoami
    HF_USERNAME = whoami(token=os.environ["HF_TOKEN"])["name"]
DATA_REPO = f"{HF_USERNAME}/okanka-mini-data"
CKPT_REPO = f"{HF_USERNAME}/okanka-mini-checkpoints"
FINAL_REPO = f"{HF_USERNAME}/okanka-mini"
print("HF user:", HF_USERNAME)

# --- code ---
gh = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
CODE = "/kaggle/working/okanka-mini"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull"], check=False)
os.chdir(CODE)
sys.path.insert(0, CODE)

# --- python deps (torch comes with the Kaggle image) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken>=0.11.0", "rustbpe>=0.1.0", "pyarrow>=21.0.0",
                "filelock", "psutil", "huggingface_hub>=0.34.0", "wandb"], check=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
major, minor = (int(x) for x in torch.__version__.split(".")[:2])
if (major, minor) < (2, 6):
    # nanochat needs a recent torch (SDPA enable_gqa, F.rms_norm, compile fixes). cu126 wheels still support T4 (sm75) and P100 (sm60).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
    print("Upgraded torch -> restart the kernel (Run > Restart) and run all cells again."); raise SystemExit

# --- nanochat runtime env ---
os.environ["NANOCHAT_BASE_DIR"] = "/tmp/nanochat"           # big ephemeral disk; persistent state lives on the HF hub
os.environ["NANOCHAT_DISABLE_FA3"] = "1"                    # FA3 is Hopper/Ampere-only; T4 uses PyTorch SDPA
os.environ["OMP_NUM_THREADS"] = "1"
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap >= (8, 0):
        os.environ["NANOCHAT_DTYPE"] = "bfloat16"
    elif cap >= (7, 0):
        os.environ["NANOCHAT_DTYPE"] = "float16"            # T4/V100: fp16 tensor cores + GradScaler (nanochat built-in)
    else:
        os.environ["NANOCHAT_DTYPE"] = "float32"            # P100: no tensor cores, triton unsupported -> no compile
        os.environ["TORCHDYNAMO_DISABLE"] = "1"
NGPU = max(1, torch.cuda.device_count())
print("NANOCHAT_DTYPE =", os.environ.get("NANOCHAT_DTYPE"), "| GPUs:", NGPU)
os.makedirs(os.environ["NANOCHAT_BASE_DIR"], exist_ok=True)

def run(cmd):
    """Run a shell command, streaming its output into the notebook; raise on failure."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed ({rc}): {cmd}")

MAKE_PUBLIC = True

In [ ]:
from okanka import hf_sync
s = hf_sync.pull(CKPT_REPO, f"chatsft_checkpoints/d{DEPTH}", os.environ["NANOCHAT_BASE_DIR"])
assert s is not None, "run notebook 03 first"
import torch
from nanochat.common import autodetect_device_type
from nanochat.checkpoint_manager import load_model
from nanochat.engine import Engine
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, tokenizer, meta = load_model("sft", device, phase="eval", model_tag=f"d{DEPTH}")
engine = Engine(model, tokenizer)
bos = tokenizer.get_bos_token_id()
U0, U1, A0, A1 = (tokenizer.encode_special(t) for t in ["<|user_start|>", "<|user_end|>", "<|assistant_start|>", "<|assistant_end|>"])
def ask(q, max_tokens=200, temperature=0.6):
    ids = [bos, U0] + tokenizer.encode(q) + [U1, A0]
    out, _ = engine.generate_batch(ids, num_samples=1, max_tokens=max_tokens, temperature=temperature, top_k=50)
    return tokenizer.decode(out[0][len(ids):])
for q in ["Merhaba! Sen kimsin?", "Seni kim yaptı?", "ChatGPT misin?", "Türkiye'nin başkenti neresidir?",
          "İstanbul hakkında kısa bilgi verir misin?", "Bana kısa bir şiir yaz.", "Fotosentez nedir?",
          "İnternete erişimin var mı?", "2 + 2 kaç eder?", "Sağlıklı beslenme için üç öneri ver."]:
    print("USER:", q); print("OKANKA:", ask(q)); print("-" * 80)

In [ ]:
# validation bits-per-byte on the held-out FineWeb-2 test shard (lower is better; tokenizer-independent)
from huggingface_hub import hf_hub_download
DATA_DIR = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "base_data_tr"); os.makedirs(DATA_DIR, exist_ok=True)
for f in ["shard_00000.parquet", "shard_99999.parquet"]:
    shutil.copy(hf_hub_download(DATA_REPO, f, repo_type="dataset", token=os.environ["HF_TOKEN"]), os.path.join(DATA_DIR, f))
from nanochat.dataloader import tokenizing_distributed_data_loader_bos_bestfit
from nanochat.loss_eval import evaluate_bpb
from nanochat.tokenizer import get_token_bytes
B, T = 4, model.config.sequence_len
loader = tokenizing_distributed_data_loader_bos_bestfit(tokenizer, B, T, "val", device=device.type)
print("val bpb:", evaluate_bpb(model, loader, 50, get_token_bytes(device=device)))

In [ ]:
run(f"python -m okanka.export_model --source sft --model-tag d{DEPTH} --out /tmp/export/okanka-mini --push {FINAL_REPO}" + ("" if MAKE_PUBLIC else " --private"))
print(f"https://huggingface.co/{FINAL_REPO}")